# Notebook 37 -- 30-day Sequential Degradation Tracking

720 consecutive 2-hour windows, linear alpha/eta_sep/beta_r decay.
SBI vs EKF comparison with coverage bands.


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle

from cstr_sbi.luyben.scenarios import generate_degradation_stream
from cstr_sbi.luyben.summaries import compute_summary_statistics
from cstr_sbi.luyben.inference import sample_posterior
from cstr_sbi.luyben.ekf import run_ekf_on_window
from cstr_sbi.luyben.priors import PARAM_NAMES

with open('../results/luyben_posterior.pkl', 'rb') as f:
    posterior = pickle.load(f)['posterior']

print('Generating 30-day degradation stream (720 windows) ...')
stream = generate_degradation_stream(seed=0)
print(f'Generated {len(stream)} windows')


In [ ]:
param_names_list = list(PARAM_NAMES)
n_params = 8
n_win = len(stream)

sbi_means = np.zeros((n_win, n_params))
sbi_lo90  = np.zeros((n_win, n_params))
sbi_hi90  = np.zeros((n_win, n_params))
ekf_means = np.zeros((n_win, n_params))
ekf_std   = np.zeros((n_win, n_params))
theta_true_arr = np.zeros((n_win, n_params))
t_starts = np.zeros(n_win)

for i, w in enumerate(stream):
    obs = np.asarray(w['obs'][0])   # (120, 8)
    t   = np.asarray(w['t'])         # (120,)
    theta_true_arr[i] = np.asarray(w['theta_true'])
    t_starts[i] = w['t_start'] / 60.0  # hours

    # SBI
    s = np.asarray(compute_summary_statistics(jnp.array(obs), jnp.array(t)))
    samples = sample_posterior(posterior, s, n_samples=2000)
    sbi_means[i] = np.mean(samples, axis=0)
    sbi_lo90[i]  = np.percentile(samples, 5,  axis=0)
    sbi_hi90[i]  = np.percentile(samples, 95, axis=0)

    # EKF
    ekf_res = run_ekf_on_window(obs, t)
    ekf_means[i] = ekf_res['final_mean']
    ekf_std[i]   = ekf_res['final_std']

    if (i+1) % 100 == 0:
        print(f'  {i+1}/{n_win} windows processed')


In [ ]:
# Plot alpha and eta_sep tracking (most interesting parameters)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for ax_idx, (param_idx, pname) in enumerate([(0, 'alpha'), (2, 'eta_sep')]):
    ax = axes[ax_idx]
    ax.plot(t_starts/24, theta_true_arr[:, param_idx], 'k-', lw=1.5, label='True')
    ax.plot(t_starts/24, sbi_means[:, param_idx], 'C0-', lw=1, label='SBI mean')
    ax.fill_between(t_starts/24, sbi_lo90[:, param_idx], sbi_hi90[:, param_idx],
                    alpha=0.3, color='C0', label='SBI 90% CI')
    ax.plot(t_starts/24, ekf_means[:, param_idx], 'C2--', lw=1, label='EKF mean')
    ax.fill_between(t_starts/24,
                    ekf_means[:, param_idx] - 1.645*ekf_std[:, param_idx],
                    ekf_means[:, param_idx] + 1.645*ekf_std[:, param_idx],
                    alpha=0.2, color='C2', label='EKF 90% CI')
    ax.set_ylabel(pname); ax.legend(loc='upper right'); ax.grid(alpha=0.3)
axes[1].set_xlabel('Time [days]')
fig.suptitle('30-day tracking: SBI vs EKF')
plt.show()

# MAE
for j, pname in enumerate(param_names_list[:4]):
    sbi_mae = np.mean(np.abs(sbi_means[:, j] - theta_true_arr[:, j]))
    ekf_mae = np.mean(np.abs(ekf_means[:, j] - theta_true_arr[:, j]))
    print(f'{pname:10s}: SBI MAE={sbi_mae:.4f}  EKF MAE={ekf_mae:.4f}')
